In [ ]:
import json
from datetime import datetime
from ultralytics import YOLO
import cv2
import numpy as np

# 1. Configuración
ruta_mi_modelo = "runs/detect/wheat_fast_v1/weights/best.pt"
model = YOLO(ruta_mi_modelo)
image_path = "trigo_maduro.jpg"
img = cv2.imread(image_path)

# 2. Ejecutar detección
results = model(image_path, conf=0.25)

verdes = 0
intermedios = 0
doradas = 0
detalles_espigas = []

for r in results:
    for i, box in enumerate(r.boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        espiga_crop = img[y1:y2, x1:x2]
        if espiga_crop.size == 0: continue

        hsv_espiga = cv2.cvtColor(espiga_crop, cv2.COLOR_BGR2HSV)
        hue_medio = np.mean(hsv_espiga[:, :, 0])
        
        # ESCALA REFINADA DE COLOR (HUE):
        # Verde puro: > 45
        # Amarillo/Verdoso (Intermedio): 30 a 45
        # Dorado/Marrón (Maduro): < 30
        
        if hue_medio > 45:
            estado_ind = "verde"
            verdes += 1
            color_box = (0, 255, 0)      # Verde
        elif 30 <= hue_medio <= 45:
            estado_ind = "intermedio"
            intermedios += 1
            color_box = (0, 255, 255)    # Amarillo brillante
        else:
            estado_ind = "dorada"
            doradas += 1
            color_box = (0, 165, 255)    # Naranja/Dorado

        detalles_espigas.append({
            "id": i,
            "estado": estado_ind,
            "hue_valor": round(float(hue_medio), 2)
        })
        
        cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)

# 3. LÓGICA DE DECISIÓN AGRONÓMICA (Basada en porcentajes)
total = verdes + intermedios + doradas
p_madurez = ((doradas + (intermedios * 0.5)) / total * 100) if total > 0 else 0

if p_madurez < 25:
    estado_campo = "CRECIMIENTO ACTIVO (VERDE)"
elif 25 <= p_madurez < 50:
    estado_campo = "MADURACIÓN INICIAL (GRANO LECHOSO)"
elif 50 <= p_madurez < 85:
    estado_campo = "MADURACIÓN AVANZADA (GRANO PASTOSO)"
else:
    estado_campo = "LISTO PARA COSECHAR (SECADO FINAL)"

# 4. GENERAR JSON COMPLETO
reporte_data = {
    "resumen": {
        "total": total,
        "conteo": {"verdes": verdes, "intermedios": intermedios, "doradas": doradas},
        "indice_madurez": round(p_madurez, 2),
        "estado_agronomico": estado_campo
    },
    "detecciones": detalles_espigas
}

with open('reporte_vegetai.json', 'w', encoding='utf-8') as f:
    json.dump(reporte_data, f, ensure_ascii=False, indent=4)

cv2.imwrite("reporte_madurez_vegetai.jpg", img)
print(f"✅ Análisis finalizado: {estado_campo}")


image 1/1 c:\Users\Administrador\Desktop\repositorios_agc\repo_agc\repo_vegetAIbles\VegetAI-labels\trigo_maduro.jpg: 352x512 2 whds, 314.5ms
Speed: 4.6ms preprocess, 314.5ms inference, 6.2ms postprocess per image at shape (1, 3, 352, 512)
✅ Análisis finalizado: LISTO PARA COSECHAR (SECADO FINAL)


: 